# AMEX Enterprise Credit Risk Platform
## Notebook 33 — Phase 2, Problem 3: Expected Credit Loss (IFRS9/CECL) — Financial-Impact Reporting & Packaging
### Problem Statement 3 of 14 (notebook 4 of 4)
### CRISP-DM stage: Deployment (business handoff)

Fourth and final notebook for Problem 3. Translates Notebooks 30-32's real, validated IFRS9/CECL ECL work
into a financial-impact package: a Word report, a standalone interactive HTML dashboard, and a colorful
Excel workbook with a real Table + AutoFilter dropdowns + conditional formatting + a linked chart — plus
SMART suggestions for every organizational level from the Basel desk to the CFO. Reuses the exact template
proven in Notebook 29 (Problem 4).

**Honesty boundary:** the ECL totals (flat/IFRS9/CECL), the macro-scenario stress range, and the statistical
validation figures below are all computed live from Notebooks 30-32's real, already-validated results — not
recomputed or re-derived here. Three financial-planning inputs are explicit, editable `ASSUMPTION`s this
dataset cannot supply: an analyst-hours-saved-per-cycle estimate, a fully-loaded analyst hourly cost, and a
one-time implementation cost / validation cadence used only for the operational-efficiency ROI estimate.
The reserve and capital-impact figures are reported as real provisioning changes, not folded into the ROI
ratio as if they were free cash — a provisioning increase is a capital decision, not a benefit to monetize.

**Deliverables:** `ECL_Financial_Impact_Report.docx`, `ecl_financial_impact_dashboard.html`,
`AMEX_Problem3_Financial_Impact_Workbook.xlsx`, plus inline charts and tables.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 30/31/32's REAL RESULTS
# =============================================================================
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 30/31/32's Real Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P3_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem3_Expected_Credit_Loss_IFRS9_CECL"
ARTIFACTS_DIR = P3_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PILLAR_DIRS = {
    "p3_policy": P3_ROOT / "01_ECL_Policy",
    "p3_modeling": P3_ROOT / "02_ECL_Modeling",
    "p3_validation_deployment": P3_ROOT / "03_Validation_Deployment",
    "p3_reporting_packaging": P3_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

ECL_POLICY_PATH = PILLAR_DIRS["p3_policy"] / "ecl_policy.json"
COMPARISON_PATH = PILLAR_DIRS["p3_modeling"] / "ecl_standard_comparison.csv"
NB31_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_31_summary.json"
SENSITIVITY_PATH = PILLAR_DIRS["p3_validation_deployment"] / "p3_macro_sensitivity.csv"
READINESS_PATH = PILLAR_DIRS["p3_validation_deployment"] / "p3_deployment_readiness_checklist.csv"
NB32_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_32_summary.json"
for _p, _fix in [
    (ECL_POLICY_PATH, "run Notebook 30 first."),
    (COMPARISON_PATH, "run Notebook 31 first."),
    (NB31_SUMMARY_PATH, "run Notebook 31 first."),
    (SENSITIVITY_PATH, "run Notebook 32 first."),
    (READINESS_PATH, "run Notebook 32 first."),
    (NB32_SUMMARY_PATH, "run Notebook 32 first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(ECL_POLICY_PATH, "r", encoding="utf-8") as f:
    ECL_POLICY = json.load(f)
with open(NB31_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB31_SUMMARY = json.load(f)
with open(NB32_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB32_SUMMARY = json.load(f)

if not NB32_SUMMARY.get("self_test_passed", False):
    raise RuntimeError(
        "Notebook 32's standalone-calculator self-test did NOT pass on its own run -- Problem 3's ECL "
        "engine is not deployment-ready. Fix: re-run Notebooks 30-32 and resolve the self-test failure "
        "before packaging a financial-impact report on top of it."
    )

EAD_PER_ACCOUNT_USD = ECL_POLICY["ead_per_account_usd"]
BASELINE_LGD_FLAT = ECL_POLICY["flat_lgd_comparison_baseline"]["lgd_flat"]
NB08_TOTAL_ECL_USD = ECL_POLICY["flat_lgd_comparison_baseline"]["total_ecl_usd"]
N_HOLDOUT = NB31_SUMMARY["n_holdout_customers"]
CHAMPION_NAME = NB31_SUMMARY["champion_pd_model"]

print(f"Real holdout population (Notebook 31)     : {N_HOLDOUT:,} customers")
print(f"Champion PD model (Problem 1, real)        : {CHAMPION_NAME}")
print(f"Baseline flat LGD (Notebook 08, real)      : {BASELINE_LGD_FLAT:.2%}")
print(f"EAD per account (Notebook 08, ASSUMPTION)  : ${EAD_PER_ACCOUNT_USD:,}")
print(f"Notebook 32 deployment self-test           : {'PASSED' if NB32_SUMMARY['self_test_passed'] else 'FAILED'}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
except ImportError:
    missing.append("python-docx")
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.formatting.rule import ColorScaleRule
    from openpyxl.chart import BarChart, Reference
except ImportError:
    missing.append("openpyxl")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: FINANCIAL PLANNING ASSUMPTIONS (EXPLICIT, EDITABLE)
# =============================================================================
_section("SECTION 3: Financial Planning Assumptions (Explicit, Editable)")

# --- This dataset has no real staffing/cost data for the model-risk and finance teams that run
#     dual-standard (IFRS9/CECL) reserve reconciliation and statistical re-validation each cycle.
#     Every figure below is a stated, editable ASSUMPTION -- edit to your institution's own numbers.
#     Nothing here is fabricated as if it were measured, and none of it is folded into the real
#     reserve-impact figures computed in Sections 4-6 below. ---
FINANCIAL_ASSUMPTIONS = {
    "ead_per_account_usd": {"value": EAD_PER_ACCOUNT_USD, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "baseline_flat_lgd": {"value": BASELINE_LGD_FLAT, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "analyst_hours_saved_per_cycle": {
        "value": 250,
        "source": "ASSUMPTION -- illustrative hours saved per quarterly cycle vs a manual dual-standard "
                   "(IFRS9 3-stage + CECL lifetime) reserve reconciliation, statistical re-validation "
                   "(chi-square/z-test/PSI), and audit-trail assembly across two accounting standards; "
                   "edit to your institution's own process-timing data.",
    },
    "analyst_hourly_cost_usd": {
        "value": 100,
        "source": "ASSUMPTION -- illustrative fully-loaded hourly cost for model-risk/quant-analyst time; "
                   "edit to your institution's actual cost figure.",
    },
    "implementation_cost_usd": {
        "value": 70_000,
        "source": "ASSUMPTION -- illustrative one-time build/validate/deploy cost for a dual-framework "
                   "(IFRS9+CECL) ECL engine with statistical validation and macro-sensitivity tooling -- "
                   "larger than Problem 4's single-framework LGD engine; edit to your institution's actual "
                   "project cost.",
    },
    "annual_validation_cycles": {
        "value": 4,
        "source": "ASSUMPTION -- how many times per year this ECL pipeline is re-run and re-validated against "
                   "a holdout-sized population (illustrative quarterly cadence); edit to your institution's "
                   "actual monitoring frequency.",
    },
}
ANALYST_HOURS_SAVED = FINANCIAL_ASSUMPTIONS["analyst_hours_saved_per_cycle"]["value"]
ANALYST_HOURLY_COST_USD = FINANCIAL_ASSUMPTIONS["analyst_hourly_cost_usd"]["value"]
IMPLEMENTATION_COST_USD = FINANCIAL_ASSUMPTIONS["implementation_cost_usd"]["value"]
ANNUAL_VALIDATION_CYCLES = FINANCIAL_ASSUMPTIONS["annual_validation_cycles"]["value"]

assumptions_path = ARTIFACTS_DIR / "p3_financial_assumptions.json"
with open(assumptions_path, "w", encoding="utf-8") as f:
    json.dump(FINANCIAL_ASSUMPTIONS, f, indent=2)
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    print(f"  {_k}: {_v['value']}  ({_v['source']})")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: RESERVE & CAPITAL IMPACT -- FLAT VS IFRS9-STAGED VS CECL (REAL)
# =============================================================================
_section("SECTION 4: Reserve & Capital Impact -- Flat vs IFRS9-Staged vs CECL (Real)")

comparison_df = pd.read_csv(COMPARISON_PATH)
# --- Display-rounded to whole-cent USD -- these are already the real pipeline outputs (Notebook 31);
#     rounding to 2dp here only affects presentation (Word/console), never the values used to compute
#     the reserve/CECL/stress deltas below (all still derived from the real, full-precision figures). ---
comparison_df["total_ecl_usd"] = comparison_df["total_ecl_usd"].round(2)
comparison_df["delta_vs_notebook_08_usd"] = comparison_df["delta_vs_notebook_08_usd"].round(2)


def _get_ecl(_substr: str) -> float:
    _rows = comparison_df[comparison_df["methodology"].str.contains(_substr, regex=False)]
    if len(_rows) != 1:
        raise RuntimeError(f"Expected exactly 1 row containing '{_substr}' in {COMPARISON_PATH.name}, "
                            f"found {len(_rows)}. Fix: re-run Notebook 31.")
    return float(_rows.iloc[0]["total_ecl_usd"])


LOSS_FLAT_USD = _get_ecl("Notebook 08")
LOSS_IFRS9_USD = _get_ecl("IFRS9 staged")
LOSS_CECL_USD = _get_ecl("CECL lifetime")
RESERVE_CHANGE_IFRS9_VS_FLAT_USD = LOSS_IFRS9_USD - LOSS_FLAT_USD
_direction = "MORE" if RESERVE_CHANGE_IFRS9_VS_FLAT_USD > 0 else "LESS"
_interpretation = (
    "the flat-LGD, outcome-based-staging approach was UNDER-recognizing loss on this real holdout -- "
    "moving to tier-differentiated LGD with outcome-free staging surfaces regulatory/under-provisioning "
    "risk the flat approach was hiding."
    if RESERVE_CHANGE_IFRS9_VS_FLAT_USD > 0 else
    "the flat-LGD, outcome-based-staging approach was OVER-recognizing loss on this real holdout -- the "
    "tier-differentiated, outcome-free approach frees capital that was being conservatively over-reserved."
)
print(comparison_df.to_string(index=False))
print(f"\nTotal loss -- Notebook 08 flat-LGD baseline          : ${LOSS_FLAT_USD:,.0f}")
print(f"Total loss -- IFRS9 staged, tier-LGD (Notebook 31)    : ${LOSS_IFRS9_USD:,.0f}")
print(f"Total loss -- CECL lifetime-for-all (Notebook 31)     : ${LOSS_CECL_USD:,.0f}")
print(f"Reserve change, IFRS9 vs flat baseline                : ${RESERVE_CHANGE_IFRS9_VS_FLAT_USD:,.0f} ({_direction} recognized)")
print(f"Interpretation: {_interpretation}")
print(f"(Measured on this real holdout of {N_HOLDOUT:,} customers -- scale by your own "
      f"total-portfolio-size ratio for an institution-wide figure.)")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: CECL ADOPTION IMPACT -- LIFETIME-FOR-ALL VS STAGED (REAL)
# =============================================================================
_section("SECTION 5: CECL Adoption Impact -- Lifetime-for-All vs Staged (Real)")

CECL_VS_IFRS9_DELTA_USD = LOSS_CECL_USD - LOSS_IFRS9_USD
_cecl_pct = (CECL_VS_IFRS9_DELTA_USD / LOSS_IFRS9_USD * 100) if LOSS_IFRS9_USD else float("nan")
print(f"Additional day-one reserve required under CECL vs IFRS9: ${CECL_VS_IFRS9_DELTA_USD:,.0f} ({_cecl_pct:+.1f}%)")
print("This is the real, expected effect of CECL requiring lifetime expected losses for the ENTIRE "
      "portfolio from day one (US GAAP), versus IFRS9 reserving lifetime losses only for Stage 2/3 "
      "accounts (IASB). A US GAAP filer transitioning from an IFRS9-style framework should plan for "
      "this as a one-time day-one reserve build.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: MACRO STRESS BUFFER -- DOWNSIDE VS BASELINE VS UPSIDE (REAL)
# =============================================================================
_section("SECTION 6: Macro Stress Buffer -- Downside vs Baseline vs Upside (Real)")

sensitivity_df = pd.read_csv(SENSITIVITY_PATH)
sensitivity_df["total_ecl_ifrs9_usd"] = sensitivity_df["total_ecl_ifrs9_usd"].round(2)  # display-only, see note above


def _get_scenario_ecl(_scenario: str) -> float:
    _rows = sensitivity_df[sensitivity_df["scenario"] == _scenario]
    if len(_rows) != 1:
        raise RuntimeError(f"Expected exactly 1 '{_scenario}' row in {SENSITIVITY_PATH.name}, "
                            f"found {len(_rows)}. Fix: re-run Notebook 32.")
    return float(_rows.iloc[0]["total_ecl_ifrs9_usd"])


ECL_UPSIDE_USD = _get_scenario_ecl("Upside")
ECL_BASELINE_USD = _get_scenario_ecl("Baseline")
ECL_DOWNSIDE_USD = _get_scenario_ecl("Downside")
STRESS_BUFFER_USD = ECL_DOWNSIDE_USD - ECL_BASELINE_USD
UPSIDE_RELIEF_USD = ECL_BASELINE_USD - ECL_UPSIDE_USD

print(sensitivity_df.to_string(index=False))
print(f"\nBaseline-scenario IFRS9 ECL (real, deterministic)  : ${ECL_BASELINE_USD:,.0f}")
print(f"Downside-scenario IFRS9 ECL (real, deterministic)  : ${ECL_DOWNSIDE_USD:,.0f}")
print(f"Recommended stress capital buffer (Downside - Baseline): ${STRESS_BUFFER_USD:,.0f}")
print(f"Upside-scenario relief (Baseline - Upside)             : ${UPSIDE_RELIEF_USD:,.0f}")
print("A CFO planning capital adequacy under adverse macro conditions should hold at least the stress "
      "buffer above the probability-blended reserve booked in Notebook 31.")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: STATISTICAL VALIDATION & DEPLOYMENT READINESS RECAP (REAL)
# =============================================================================
_section("SECTION 7: Statistical Validation & Deployment Readiness Recap (Real)")

readiness_df = pd.read_csv(READINESS_PATH)
print(readiness_df.to_string(index=False))
print(f"\nChi-square p-value (stage vs default, real)   : {NB32_SUMMARY['chi_square_p_value']:.2e}")
print(f"Cramer's V effect size (real)                  : {NB32_SUMMARY['cramers_v']:.4f}")
print(f"Two-proportion z-test p-value (real)           : {NB32_SUMMARY['z_test_p_value']:.2e}")
print(f"Split-half PSI (real, {NB32_SUMMARY['psi_verdict']}){' ' * max(0, 1)}          : {NB32_SUMMARY['psi']:.4f}")
print(f"Standalone calculator self-test (real)         : "
      f"{NB32_SUMMARY['self_test_hard_mismatches']} hard mismatches, "
      f"{NB32_SUMMARY['self_test_boundary_ties']} boundary ties, of "
      f"{NB32_SUMMARY['self_test_customers_checked']} customers checked -- "
      f"{'PASSED' if NB32_SUMMARY['self_test_passed'] else 'FAILED'}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: ROI, INVESTMENT & PAYBACK (OPERATIONAL EFFICIENCY ONLY)
# =============================================================================
_section("SECTION 8: ROI, Investment & Payback (Operational Efficiency Only)")

# --- Deliberately NOT folding the real reserve-change figures above into this ROI ratio: a
#     provisioning increase/decrease is a capital-adequacy decision, not free cash a project
#     "earns". The ROI below is scoped ONLY to the operational efficiency of automating a
#     dual-standard reserve reconciliation and statistical re-validation that would otherwise be
#     done manually each cycle -- an honest, narrower claim than Notebook 29's loss-prevention ROI. ---
EFFICIENCY_BENEFIT_PER_CYCLE_USD = ANALYST_HOURS_SAVED * ANALYST_HOURLY_COST_USD
ANNUAL_EFFICIENCY_BENEFIT_USD = EFFICIENCY_BENEFIT_PER_CYCLE_USD * ANNUAL_VALIDATION_CYCLES
ROI_PCT = ((ANNUAL_EFFICIENCY_BENEFIT_USD - IMPLEMENTATION_COST_USD) / IMPLEMENTATION_COST_USD) * 100 if IMPLEMENTATION_COST_USD else None
PAYBACK_MONTHS = (IMPLEMENTATION_COST_USD / (ANNUAL_EFFICIENCY_BENEFIT_USD / 12)) if ANNUAL_EFFICIENCY_BENEFIT_USD > 0 else None

print(f"Amount invested (ASSUMPTION, one-time)          : ${IMPLEMENTATION_COST_USD:,.0f}")
print(f"Efficiency benefit per cycle (ASSUMPTION hours x cost): ${EFFICIENCY_BENEFIT_PER_CYCLE_USD:,.0f}")
print(f"Estimated annual efficiency benefit              : ${ANNUAL_EFFICIENCY_BENEFIT_USD:,.0f} "
      f"(= per-cycle benefit x {ANNUAL_VALIDATION_CYCLES} cycles/year, ASSUMPTION)")
print(f"Estimated ROI (Year 1, efficiency only)          : {ROI_PCT:,.0f}%" if ROI_PCT is not None else "n/a")
print(f"Estimated payback period                         : {PAYBACK_MONTHS:.1f} months" if PAYBACK_MONTHS else "n/a")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: SMART SUGGESTIONS -- BOTTOM TO TOP MANAGEMENT
# =============================================================================
_section("SECTION 9: SMART Suggestions -- Bottom to Top Management")

SMART_SUGGESTIONS = [
    {"org_level": "Risk / Quant Analyst",
     "suggestion": f"Re-run Notebooks 30-32 each time Problem 1's champion model or Problem 4's severity "
                   f"tiers refresh; monitor the split-half PSI (last measured {NB32_SUMMARY['psi']:.4f}, "
                   f"'{NB32_SUMMARY['psi_verdict']}') and re-validate staging if it exceeds 0.10 for two "
                   f"consecutive cycles."},
    {"org_level": "Model Risk / Compliance (SR 11-7)",
     "suggestion": f"File this notebook's chi-square (p={NB32_SUMMARY['chi_square_p_value']:.1e}) and "
                   f"z-test (p={NB32_SUMMARY['z_test_p_value']:.1e}) results, plus the standalone "
                   f"calculator's {NB32_SUMMARY['self_test_customers_checked']}/{NB32_SUMMARY['self_test_customers_checked']} "
                   f"self-test pass, with the model's annual validation packet; re-certify before any change "
                   f"to the frozen ecl_scoring_bundle.json."},
    {"org_level": "Finance / Provisioning",
     "suggestion": f"Update the quarterly provisioning workbook to book IFRS9-staged reserve of "
                   f"${LOSS_IFRS9_USD:,.0f} (a ${abs(RESERVE_CHANGE_IFRS9_VS_FLAT_USD):,.0f} change from the "
                   f"flat-LGD baseline); if reporting under US GAAP, additionally plan for the "
                   f"${CECL_VS_IFRS9_DELTA_USD:,.0f} CECL day-one reserve build."},
    {"org_level": "Regulatory Reporting / Basel Desk",
     "suggestion": f"Reconcile this notebook's tier-LGD, outcome-free-staged ECL against Notebook 08's "
                   f"flat-LGD Basel capital figures each cycle -- both trace to the same real champion PD "
                   f"model ('{CHAMPION_NAME}'); document the ${abs(RESERVE_CHANGE_IFRS9_VS_FLAT_USD):,.0f} "
                   f"reconciling difference in the regulatory capital workpapers."},
    {"org_level": "Executive / CFO",
     "suggestion": f"Hold a stress capital buffer of at least ${STRESS_BUFFER_USD:,.0f} above the "
                   f"probability-blended IFRS9 reserve for adverse macro conditions (Notebook 32's Downside "
                   f"scenario); approve the ${IMPLEMENTATION_COST_USD:,.0f} automation investment given an "
                   f"estimated {PAYBACK_MONTHS:.1f}-month payback and {ROI_PCT:,.0f}% Year-1 efficiency ROI."},
]
smart_df = pd.DataFrame(SMART_SUGGESTIONS)
smart_path = PILLAR_DIRS["p3_reporting_packaging"] / "p3_smart_suggestions.csv"
smart_df.to_csv(smart_path, index=False)
for _row in SMART_SUGGESTIONS:
    print(f"[{_row['org_level']}]\n  {_row['suggestion']}\n")
print(f"\u2705 Saved -> {smart_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: INLINE CHARTS
# =============================================================================
_section("SECTION 10: Inline Charts")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#C9A227", "surface": "#FFFFFF"}

fig1, ax1 = plt.subplots(figsize=(7.5, 5), dpi=150)
_labels = ["Flat LGD\n(Notebook 08)", "IFRS9 Staged\n(Notebook 31)", "CECL Lifetime\n(Notebook 31)"]
_values = [LOSS_FLAT_USD, LOSS_IFRS9_USD, LOSS_CECL_USD]
_bars = ax1.bar(_labels, _values, color=[VIZ["muted"], VIZ["accent"], VIZ["gold"]])
for _b, _v in zip(_bars, _values):
    ax1.text(_b.get_x() + _b.get_width() / 2, _v, f"${_v:,.0f}", ha="center", va="bottom", fontsize=10)
ax1.set_ylabel("Recognized ECL, real holdout (USD)")
ax1.set_title("Problem 3: Flat LGD vs IFRS9 Staged vs CECL Lifetime")
fig1.tight_layout()
chart1_path = PILLAR_DIRS["p3_reporting_packaging"] / "ecl_standard_comparison_final_chart.png"
fig1.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(7.5, 5), dpi=150)
_scenarios = sensitivity_df["scenario"].tolist()
_scen_values = sensitivity_df["total_ecl_ifrs9_usd"].tolist()
_colors = [VIZ["muted"] if s == "Upside" else (VIZ["gold"] if s == "Baseline" else VIZ["accent"]) for s in _scenarios]
_bars2 = ax2.bar(_scenarios, _scen_values, color=_colors)
for _b, _v in zip(_bars2, _scen_values):
    ax2.text(_b.get_x() + _b.get_width() / 2, _v, f"${_v:,.0f}", ha="center", va="bottom", fontsize=10)
ax2.axhline(ECL_BASELINE_USD, color=VIZ["ink"], linestyle="--", linewidth=1,
            label=f"Baseline: ${ECL_BASELINE_USD:,.0f}")
ax2.set_ylabel("Total IFRS9 ECL, deterministic per scenario (USD)")
ax2.set_title(f"Problem 3: Macro Stress Buffer (Downside - Baseline = ${STRESS_BUFFER_USD:,.0f})")
ax2.legend()
fig2.tight_layout()
chart2_path = PILLAR_DIRS["p3_reporting_packaging"] / "macro_stress_buffer_final_chart.png"
fig2.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig2)

print(f"\u2705 Saved -> {chart1_path.name}, {chart2_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: WORD REPORT -- ECL_Financial_Impact_Report.docx
# =============================================================================
_section("SECTION 11: Word Report -- ECL_Financial_Impact_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 2, Problem 3: Expected Credit Loss (IFRS9/CECL) -- Financial Impact Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Executive Summary", level=1)
doc.add_paragraph(
    f"An outcome-free IFRS9 staging rubric and tier-differentiated LGD (Notebooks 30-31), validated "
    f"statistically and packaged as a self-tested standalone calculator (Notebook 32), was applied to "
    f"{N_HOLDOUT:,} real holdout customers. Compared to Notebook 08's flat-LGD, outcome-based-staging "
    f"approach, this changes recognized loss by ${RESERVE_CHANGE_IFRS9_VS_FLAT_USD:,.0f} -- {_interpretation} "
    f"Reporting under CECL instead of IFRS9 would require an additional ${CECL_VS_IFRS9_DELTA_USD:,.0f} "
    f"day-one reserve. A CFO should hold a stress capital buffer of at least ${STRESS_BUFFER_USD:,.0f} for "
    f"adverse macro conditions. Automating the dual-standard reconciliation and statistical re-validation "
    f"this pipeline performs is estimated to pay back its ${IMPLEMENTATION_COST_USD:,.0f} implementation "
    f"cost in {PAYBACK_MONTHS:.1f} months."
)

_add_heading(doc, "2. Reserve & Capital Impact: Flat vs IFRS9 vs CECL", level=1)
_add_table_from_df(doc, comparison_df)
doc.add_paragraph(f"Reserve change, IFRS9 vs flat baseline: ${RESERVE_CHANGE_IFRS9_VS_FLAT_USD:,.0f}  |  "
                   f"CECL vs IFRS9 day-one delta: ${CECL_VS_IFRS9_DELTA_USD:,.0f}")

_add_heading(doc, "3. Macro Stress Buffer", level=1)
_add_table_from_df(doc, sensitivity_df)
_add_kv_table(doc, {"recommended_stress_buffer_usd": f"${STRESS_BUFFER_USD:,.0f}",
                     "upside_relief_usd": f"${UPSIDE_RELIEF_USD:,.0f}"})

_add_heading(doc, "4. Statistical Validation & Deployment Readiness", level=1)
_add_table_from_df(doc, readiness_df)

_add_heading(doc, "5. ROI, Investment & Payback (Operational Efficiency)", level=1)
_add_kv_table(doc, {"amount_invested_usd": f"${IMPLEMENTATION_COST_USD:,.0f}",
                     "annual_efficiency_benefit_usd": f"${ANNUAL_EFFICIENCY_BENEFIT_USD:,.0f}",
                     "roi_year_1_pct": f"{ROI_PCT:,.0f}%", "payback_period_months": f"{PAYBACK_MONTHS:.1f}"})

_add_heading(doc, "6. SMART Suggestions by Organizational Level", level=1)
_add_table_from_df(doc, smart_df)

_add_heading(doc, "7. Assumptions & Sources", level=1)
_assump_df = pd.DataFrame([{"assumption": k, "value": v["value"], "source": v["source"]}
                            for k, v in FINANCIAL_ASSUMPTIONS.items()])
_add_table_from_df(doc, _assump_df)

report_path = PILLAR_DIRS["p3_reporting_packaging"] / "ECL_Financial_Impact_Report.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: EXCEL WORKBOOK -- COLORFUL, TABLE + AUTOFILTER + CONDITIONAL FORMATTING + CHART
# =============================================================================
_section("SECTION 12: Excel Workbook -- Colorful, Table + AutoFilter + Conditional Formatting + Chart")

INK = "0B1F3A"
ACCENT = "C41E3A"
GOLD = "C9A227"
LIGHT = "F2F4F8"
WHITE = "FFFFFF"
USD_FMT = '$#,##0;($#,##0);-'

_assump_rows = {k: 2 + i for i, k in enumerate(FINANCIAL_ASSUMPTIONS.keys())}
_comp_first_row, _comp_last_row = 2, 1 + len(comparison_df)
_sens_first_row, _sens_last_row = 2, 1 + len(sensitivity_df)

wb = openpyxl.Workbook()

# --- Sheet 1: Assumptions (yellow-highlighted per financial-model convention) ---
ws_assump = wb.active
ws_assump.title = "Assumptions"
ws_assump.append(["Assumption", "Value", "Source / Rationale"])
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    ws_assump.append([_k.replace("_", " ").title(), _v["value"], _v["source"]])
for _r in range(2, ws_assump.max_row + 1):
    ws_assump[f"B{_r}"].fill = PatternFill("solid", fgColor="FFFF00")
    ws_assump[f"B{_r}"].font = Font(name="Calibri", color="0000FF")  # blue = hardcoded input, per convention
    ws_assump[f"C{_r}"].alignment = Alignment(wrap_text=True, vertical="top")
ws_assump[f"B{_assump_rows['baseline_flat_lgd']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['ead_per_account_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['analyst_hourly_cost_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['implementation_cost_usd']}"].number_format = USD_FMT
ws_assump.column_dimensions["A"].width = 32
ws_assump.column_dimensions["B"].width = 14
ws_assump.column_dimensions["C"].width = 95
_tbl_assump = Table(displayName="Assumptions", ref=f"A1:C{ws_assump.max_row}")
_tbl_assump.tableStyleInfo = TableStyleInfo(name="TableStyleMedium4", showRowStripes=True)
ws_assump.add_table(_tbl_assump)

_hours_ref = f"Assumptions!$B${_assump_rows['analyst_hours_saved_per_cycle']}"
_hourly_cost_ref = f"Assumptions!$B${_assump_rows['analyst_hourly_cost_usd']}"
_cost_ref = f"Assumptions!$B${_assump_rows['implementation_cost_usd']}"
_cycles_ref = f"Assumptions!$B${_assump_rows['annual_validation_cycles']}"

# --- Sheet 2: Standard Comparison -- real pipeline outputs (not re-derivable from Assumptions,
#     each total is the result of a full ECL scoring run), delta computed as a real Excel formula. ---
ws_comp = wb.create_sheet("Standard Comparison")
ws_comp.append(["Methodology", "Total ECL (USD)", "Delta vs Notebook 08 (USD)"])
for _i, _r in comparison_df.iterrows():
    _row_n = _comp_first_row + _i
    ws_comp.append([_r["methodology"], float(_r["total_ecl_usd"]), None])
    ws_comp[f"C{_row_n}"] = f"=B{_row_n}-$B${_comp_first_row}"
for _col_letter in ("B", "C"):
    for _r in range(_comp_first_row, _comp_last_row + 1):
        ws_comp[f"{_col_letter}{_r}"].number_format = USD_FMT
ws_comp.column_dimensions["A"].width = 55
ws_comp.column_dimensions["B"].width = 18
ws_comp.column_dimensions["C"].width = 22
_tbl_comp = Table(displayName="StandardComparison", ref=f"A1:C{_comp_last_row}")
_tbl_comp.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws_comp.add_table(_tbl_comp)
ws_comp.conditional_formatting.add(
    f"B{_comp_first_row}:B{_comp_last_row}",
    ColorScaleRule(start_type="min", start_color="63BE7B", end_type="max", end_color="F8696B"))

_chart1 = BarChart()
_chart1.title = "Total ECL by Methodology (filter the table to update this chart)"
_chart1.y_axis.title = "Total ECL (USD)"
_data1 = Reference(ws_comp, min_col=2, max_col=2, min_row=1, max_row=_comp_last_row)
_cats1 = Reference(ws_comp, min_col=1, min_row=_comp_first_row, max_row=_comp_last_row)
_chart1.add_data(_data1, titles_from_data=True)
_chart1.set_categories(_cats1)
_chart1.width, _chart1.height = 18, 10
ws_comp.add_chart(_chart1, "E2")

# --- Sheet 3: Macro Sensitivity -- real per-scenario outputs, stress-buffer range as a live formula. ---
ws_sens = wb.create_sheet("Macro Sensitivity")
ws_sens.append(["Scenario", "Probability", "PD Multiplier", "Total IFRS9 ECL (USD)"])
for _i, _r in sensitivity_df.iterrows():
    ws_sens.append([_r["scenario"], float(_r["probability"]), float(_r["pd_multiplier"]),
                     float(_r["total_ecl_ifrs9_usd"])])
for _r in range(_sens_first_row, _sens_last_row + 1):
    ws_sens[f"B{_r}"].number_format = "0.0%"
    ws_sens[f"D{_r}"].number_format = USD_FMT
ws_sens.column_dimensions["A"].width = 16
ws_sens.column_dimensions["B"].width = 14
ws_sens.column_dimensions["C"].width = 14
ws_sens.column_dimensions["D"].width = 22
_tbl_sens = Table(displayName="MacroSensitivity", ref=f"A1:D{_sens_last_row}")
_tbl_sens.tableStyleInfo = TableStyleInfo(name="TableStyleMedium6", showRowStripes=True)
ws_sens.add_table(_tbl_sens)
# Matches the notebook's real definition exactly: Downside minus Baseline (NOT Max-Min/the full
# Upside-to-Downside range) -- row order is fixed by ECL_POLICY["macro_overlay"]["scenarios"]
# (Upside, Baseline, Downside), which Notebook 32 iterates in that same order to build this sheet.
_baseline_row = _sens_first_row + 1
_downside_row = _sens_first_row + 2
ws_sens["F2"] = "Stress Buffer (Downside - Baseline)"
ws_sens["G2"] = f"=D{_downside_row}-D{_baseline_row}"
ws_sens["G2"].number_format = USD_FMT
ws_sens["G2"].font = Font(bold=True)
ws_sens.column_dimensions["F"].width = 26

_chart2 = BarChart()
_chart2.title = "IFRS9 ECL by Macro Scenario"
_chart2.y_axis.title = "Total IFRS9 ECL (USD)"
_data2 = Reference(ws_sens, min_col=4, max_col=4, min_row=1, max_row=_sens_last_row)
_cats2 = Reference(ws_sens, min_col=1, min_row=_sens_first_row, max_row=_sens_last_row)
_chart2.add_data(_data2, titles_from_data=True)
_chart2.set_categories(_cats2)
_chart2.width, _chart2.height = 18, 10
ws_sens.add_chart(_chart2, "F5")

# --- Sheet 4: SMART Suggestions (real Excel Table -> native AutoFilter dropdowns) ---
ws_smart = wb.create_sheet("SMART Suggestions")
ws_smart.append(["Org Level", "Suggestion"])
for _row_data in SMART_SUGGESTIONS:
    ws_smart.append([_row_data["org_level"], _row_data["suggestion"]])
_last_row_smart = ws_smart.max_row
_tbl_smart = Table(displayName="SmartSuggestions", ref=f"A1:B{_last_row_smart}")
_tbl_smart.tableStyleInfo = TableStyleInfo(name="TableStyleMedium7", showRowStripes=True)
ws_smart.add_table(_tbl_smart)
ws_smart.column_dimensions["A"].width = 30
ws_smart.column_dimensions["B"].width = 100
for _r in range(2, _last_row_smart + 1):
    ws_smart[f"B{_r}"].alignment = Alignment(wrap_text=True, vertical="top")

# --- Sheet 5: Executive Summary (KPI cards) -- built last so formulas can reference sheets above. ---
ws_exec = wb.create_sheet("Executive Summary", 0)
wb.active = 0
ws_exec.sheet_view.showGridLines = False
ws_exec["B2"] = "AMEX RiskIQ -- Problem 3: Expected Credit Loss (IFRS9/CECL)"
ws_exec["B2"].font = Font(name="Calibri", size=16, bold=True, color=WHITE)
ws_exec["B2"].fill = PatternFill("solid", fgColor=INK)
ws_exec.merge_cells("B2:F2")
ws_exec["B3"] = "Financial Impact Summary"
ws_exec["B3"].font = Font(name="Calibri", size=11, italic=True, color=INK)
ws_exec.merge_cells("B3:F3")

_kpi_rows = [
    ("Total ECL -- Flat LGD (Notebook 08)", f"='Standard Comparison'!B{_comp_first_row}", True, LIGHT),
    ("Total ECL -- IFRS9 Staged", f"='Standard Comparison'!B{_comp_first_row + 1}", True, LIGHT),
    ("Total ECL -- CECL Lifetime", f"='Standard Comparison'!B{_comp_first_row + 2}", True, LIGHT),
    ("Reserve Change (IFRS9 - Flat)", "=D6-D5", True, GOLD),
    ("CECL Day-One Delta (CECL - IFRS9)", "=D7-D6", True, GOLD),
    ("Recommended Stress Buffer", "='Macro Sensitivity'!G2", True, ACCENT),
    ("Amount Invested (Automation)", f"=\"$\"&TEXT({_cost_ref},\"#,##0\")", True, LIGHT),
    ("Estimated Year-1 Efficiency ROI", f"{ROI_PCT:,.0f}%  (reported, see Section 8)", False, ACCENT),
    ("Estimated Payback", f"{PAYBACK_MONTHS:.1f} months  (reported, see Section 8)", False, ACCENT),
]
_row = 5
for _label, _value, _is_formula, _fill in _kpi_rows:
    ws_exec.cell(row=_row, column=2, value=_label).font = Font(name="Calibri", size=11, color=INK)
    _cell = ws_exec.cell(row=_row, column=4, value=_value)
    _cell.font = Font(name="Calibri", size=13, bold=True, color=(WHITE if _fill in (GOLD, ACCENT) else INK))
    _cell.fill = PatternFill("solid", fgColor=_fill)
    _cell.alignment = Alignment(horizontal="center", wrap_text=not _is_formula)
    if _is_formula and "Amount Invested" not in _label:
        _cell.number_format = USD_FMT
    ws_exec.merge_cells(start_row=_row, start_column=4, end_row=_row, end_column=5)
    _row += 1
ws_exec["B15"] = "Rows 5-10 recalculate live from the Standard Comparison, Macro Sensitivity, and Assumptions sheets."
ws_exec["B15"].font = Font(name="Calibri", size=9, italic=True, color="8A93A6")
ws_exec.merge_cells("B15:F15")
for _col, _w in zip("BCDEF", [34, 3, 22, 22, 3]):
    ws_exec.column_dimensions[_col].width = _w

for _ws in (ws_comp, ws_sens, ws_smart, ws_assump):
    for _cell in _ws[1]:
        _cell.font = Font(name="Calibri", bold=True, color=WHITE)
        _cell.fill = PatternFill("solid", fgColor=INK)

workbook_path = PILLAR_DIRS["p3_reporting_packaging"] / "AMEX_Problem3_Financial_Impact_Workbook.xlsx"
wb.save(str(workbook_path))
print(f"\u2705 Saved -> {workbook_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: INTERACTIVE HTML DASHBOARD
# =============================================================================
_section("SECTION 13: Interactive HTML Dashboard")

_comp_json = json.dumps(comparison_df.to_dict(orient="records"))
_sens_json = json.dumps(sensitivity_df.to_dict(orient="records"))
_smart_json = json.dumps(SMART_SUGGESTIONS)
_org_levels = sorted({r["org_level"] for r in SMART_SUGGESTIONS})

_html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Problem 3 -- ECL Financial Impact Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4"></script>
<style>
  :root { --ink:#0B1F3A; --accent:#C41E3A; --gold:#C9A227; --muted:#8A93A6; --bg:#F2F4F8; --card:#FFFFFF; }
  body { font-family: Calibri, Arial, sans-serif; background: var(--bg); color: var(--ink); margin: 0; padding: 24px; }
  h1 { font-size: 22px; margin-bottom: 4px; }
  .sub { color: var(--muted); margin-bottom: 20px; }
  .kpi-row { display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 24px; }
  .kpi { background: var(--card); border-radius: 10px; padding: 16px 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); min-width: 190px; flex: 1; }
  .kpi .label { font-size: 12px; color: var(--muted); text-transform: uppercase; }
  .kpi .value { font-size: 22px; font-weight: 700; margin-top: 4px; }
  .panel { background: var(--card); border-radius: 10px; padding: 18px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid #E4E7EE; }
  th { background: var(--ink); color: #fff; }
  select { padding: 6px 10px; border-radius: 6px; border: 1px solid var(--muted); font-size: 13px; margin-bottom: 12px; }
  canvas { max-height: 360px; }
  .grid2 { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
  @media (max-width: 900px) { .grid2 { grid-template-columns: 1fr; } }
</style>
</head>
<body>
<h1>AMEX RiskIQ -- Problem 3: Expected Credit Loss (IFRS9/CECL)</h1>
<div class="sub">Financial Impact Dashboard -- real Notebook 30-32 results, ASSUMPTION values clearly marked</div>

<div class="kpi-row">
  <div class="kpi"><div class="label">Total ECL -- Flat LGD</div><div class="value">__LOSS_FLAT__</div></div>
  <div class="kpi"><div class="label">Total ECL -- IFRS9 Staged</div><div class="value">__LOSS_IFRS9__</div></div>
  <div class="kpi"><div class="label">Total ECL -- CECL Lifetime</div><div class="value">__LOSS_CECL__</div></div>
  <div class="kpi"><div class="label">Reserve Change (IFRS9-Flat)</div><div class="value">__RESERVE_CHANGE__</div></div>
  <div class="kpi"><div class="label">Stress Buffer</div><div class="value">__STRESS_BUFFER__</div></div>
  <div class="kpi"><div class="label">Efficiency ROI / Payback</div><div class="value">__ROI__ / __PAYBACK__</div></div>
</div>

<div class="grid2">
  <div class="panel">
    <h3 style="margin-top:0">Standard Comparison</h3>
    <canvas id="compChart"></canvas>
    <table id="compTable"><thead><tr><th>Methodology</th><th>Total ECL</th><th>Delta vs Flat</th></tr></thead>
      <tbody></tbody></table>
  </div>
  <div class="panel">
    <h3 style="margin-top:0">Macro Scenario Sensitivity</h3>
    <canvas id="sensChart"></canvas>
    <table id="sensTable"><thead><tr><th>Scenario</th><th>Probability</th><th>Total IFRS9 ECL</th></tr></thead>
      <tbody></tbody></table>
  </div>
</div>

<div class="panel">
  <label for="orgFilter"><b>SMART Suggestions -- filter by organizational level</b></label><br/>
  <select id="orgFilter"></select>
  <table id="smartTable"><thead><tr><th>Org Level</th><th>Suggestion</th></tr></thead><tbody></tbody></table>
</div>

<script>
const compData = __COMP_JSON__;
const sensData = __SENS_JSON__;
const smartData = __SMART_JSON__;
const orgLevels = __ORG_LEVELS__;

const fmtUsd = v => "$" + Math.round(v).toLocaleString();
const fmtPct = v => (v * 100).toFixed(1) + "%";
const flatUsd = compData[0].total_ecl_usd;

new Chart(document.getElementById("compChart").getContext("2d"), {
  type: "bar",
  data: {
    labels: compData.map(d => d.methodology.replace("Notebook ", "NB")),
    datasets: [{ label: "Total ECL (USD)", data: compData.map(d => d.total_ecl_usd),
                 backgroundColor: ["#8A93A6", "#C41E3A", "#C9A227"] }],
  },
  options: { plugins: { legend: { display: false } },
             scales: { y: { ticks: { callback: v => "$" + v.toLocaleString() } } } },
});

new Chart(document.getElementById("sensChart").getContext("2d"), {
  type: "bar",
  data: {
    labels: sensData.map(d => d.scenario),
    datasets: [{ label: "Total IFRS9 ECL (USD)", data: sensData.map(d => d.total_ecl_ifrs9_usd),
                 backgroundColor: ["#8A93A6", "#C9A227", "#C41E3A"] }],
  },
  options: { plugins: { legend: { display: false } },
             scales: { y: { ticks: { callback: v => "$" + v.toLocaleString() } } } },
});

const compBody = document.querySelector("#compTable tbody");
compData.forEach(d => {
  const tr = document.createElement("tr");
  tr.innerHTML = `<td>${d.methodology}</td><td>${fmtUsd(d.total_ecl_usd)}</td>` +
    `<td>${fmtUsd(d.delta_vs_notebook_08_usd)}</td>`;
  compBody.appendChild(tr);
});

const sensBody = document.querySelector("#sensTable tbody");
sensData.forEach(d => {
  const tr = document.createElement("tr");
  tr.innerHTML = `<td>${d.scenario}</td><td>${fmtPct(d.probability)}</td>` +
    `<td>${fmtUsd(d.total_ecl_ifrs9_usd)}</td>`;
  sensBody.appendChild(tr);
});

function renderSmart(filterLevel) {
  const tbody = document.querySelector("#smartTable tbody");
  tbody.innerHTML = "";
  smartData.filter(r => filterLevel === "All" || r.org_level === filterLevel).forEach(r => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td>${r.org_level}</td><td>${r.suggestion}</td>`;
    tbody.appendChild(tr);
  });
}
const orgSelect = document.getElementById("orgFilter");
["All", ...orgLevels].forEach(level => {
  const opt = document.createElement("option");
  opt.value = level; opt.textContent = level;
  orgSelect.appendChild(opt);
});
orgSelect.onchange = () => renderSmart(orgSelect.value);
renderSmart("All");
</script>
</body>
</html>
"""
_html = (_html
         .replace("__LOSS_FLAT__", f"${LOSS_FLAT_USD:,.0f}")
         .replace("__LOSS_IFRS9__", f"${LOSS_IFRS9_USD:,.0f}")
         .replace("__LOSS_CECL__", f"${LOSS_CECL_USD:,.0f}")
         .replace("__RESERVE_CHANGE__", f"${RESERVE_CHANGE_IFRS9_VS_FLAT_USD:,.0f}")
         .replace("__STRESS_BUFFER__", f"${STRESS_BUFFER_USD:,.0f}")
         .replace("__ROI__", f"{ROI_PCT:,.0f}%")
         .replace("__PAYBACK__", f"{PAYBACK_MONTHS:.1f} mo")
         .replace("__COMP_JSON__", _comp_json)
         .replace("__SENS_JSON__", _sens_json)
         .replace("__SMART_JSON__", _smart_json)
         .replace("__ORG_LEVELS__", json.dumps(_org_levels)))

dashboard_path = PILLAR_DIRS["p3_reporting_packaging"] / "ecl_financial_impact_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(_html)
print(f"\u2705 Saved -> {dashboard_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION
# =============================================================================
_section("SECTION 14: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Standard comparison covers Flat/IFRS9/CECL (3 rows)", len(comparison_df) == 3)
_check("Reserve change equals IFRS9 minus flat",
       abs(RESERVE_CHANGE_IFRS9_VS_FLAT_USD - (LOSS_IFRS9_USD - LOSS_FLAT_USD)) < 1e-6)
_check("CECL delta equals CECL minus IFRS9",
       abs(CECL_VS_IFRS9_DELTA_USD - (LOSS_CECL_USD - LOSS_IFRS9_USD)) < 1e-6)
_check("CECL total >= IFRS9 total (lifetime-for-all cannot recognize less)", LOSS_CECL_USD >= LOSS_IFRS9_USD - 1e-6)
_check("Stress buffer equals Downside minus Baseline",
       abs(STRESS_BUFFER_USD - (ECL_DOWNSIDE_USD - ECL_BASELINE_USD)) < 1e-6)
_check("Downside scenario ECL >= Baseline ECL (stress should not be relief)", ECL_DOWNSIDE_USD >= ECL_BASELINE_USD - 1e-6)
_check("ROI/payback are finite, positive numbers", ROI_PCT is not None and PAYBACK_MONTHS is not None
       and PAYBACK_MONTHS > 0)
_check("Notebook 32 self-test passed (hard dependency)", NB32_SUMMARY["self_test_passed"])

_expected_files = [assumptions_path, smart_path, chart1_path, chart2_path, report_path,
                    workbook_path, dashboard_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 33 verification checks failed. See \u274c line above.")
print("\nAll Notebook 33 checks passed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: WRITE NOTEBOOK 33 SUMMARY -- PROBLEM 3 COMPLETE
# =============================================================================
_section("SECTION 15: Write Notebook 33 Summary -- Problem 3 Complete")

notebook_33_summary = {
    "notebook": "33_financial_impact_reporting_packaging", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 3, "problem_name": "Expected Credit Loss (IFRS9/CECL)",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning", "problem_3_complete": True,
    "loss_flat_lgd_usd": round(LOSS_FLAT_USD, 2), "loss_ifrs9_usd": round(LOSS_IFRS9_USD, 2),
    "loss_cecl_usd": round(LOSS_CECL_USD, 2),
    "reserve_change_ifrs9_vs_flat_usd": round(RESERVE_CHANGE_IFRS9_VS_FLAT_USD, 2),
    "cecl_vs_ifrs9_delta_usd": round(CECL_VS_IFRS9_DELTA_USD, 2),
    "macro_stress_buffer_usd": round(STRESS_BUFFER_USD, 2),
    "roi_year_1_pct": round(ROI_PCT, 1), "payback_period_months": round(PAYBACK_MONTHS, 2),
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb33_summary_path = ARTIFACTS_DIR / "notebook_33_summary.json"
with open(nb33_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_33_summary, f, indent=2)
print(f"\u2705 Saved -> {nb33_summary_path.name} (this problem's artifacts folder)")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: COMPLETION SUMMARY -- PROBLEM 3 COMPLETE, NEXT: PROBLEM 5
# =============================================================================
_section("SECTION 16: Notebook 33 Complete -- Problem 3 Complete, Next: Problem 5")

print("NOTEBOOK 33: FINANCIAL-IMPACT REPORTING & PACKAGING -- COMPLETE")
print("PROBLEM 3 (EXPECTED CREDIT LOSS, IFRS9/CECL) -- ALL 4 NOTEBOOKS COMPLETE")
print(f"  Reserve change (IFRS9 vs flat baseline)     : ${RESERVE_CHANGE_IFRS9_VS_FLAT_USD:,.0f}")
print(f"  CECL day-one delta (vs IFRS9)                : ${CECL_VS_IFRS9_DELTA_USD:,.0f}")
print(f"  Recommended macro stress buffer               : ${STRESS_BUFFER_USD:,.0f}")
print(f"  Estimated Year-1 efficiency ROI / payback      : {ROI_PCT:,.0f}% / {PAYBACK_MONTHS:.1f} months")
print(f"  Files produced                                : {len(_expected_files) + 1}")
for _p in _expected_files + [nb33_summary_path]:
    print(f"    - {_p.name}")
print("  Next: Problem 5 (Early Payment Default Detection) -- depends only on Problem 1.")
print("\n\u2705 Ready to proceed.")
